In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia
!unzip -q chest-xray-pneumonia.zip

Dataset URL: https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia
License(s): other
 99% 2.28G/2.29G [00:32<00:00, 259MB/s]
100% 2.29G/2.29G [00:33<00:00, 74.5MB/s]


In [ ]:
!ls chest_xray

chest_xray  __MACOSX  test  train  val


In [ ]:
!ls chest_xray/train

NORMAL	PNEUMONIA


In [ ]:
import os
import cv2
import numpy as np



# ---- Single folder loader ----
def load_images_from_folder(folder, flatten=True):
    """
    Loads images from a folder.
    - folder: path to NORMAL/PNEUMONIA structure
    - flatten: if True, returns 1D arrays; if False, returns (H, W, 1) for CNNs
    """
    images = []
    labels = []
    for root, dirs, files in os.walk(folder):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                image_path = os.path.join(root, file)

                # Label based on folder name
                label_name = os.path.basename(os.path.dirname(image_path))
                label = 1 if label_name.upper() == "PNEUMONIA" else 0

                # Read grayscale
                img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
                if img is None:
                    print(f"Could not read {image_path}")
                    continue

                # Resize to 64x64
                img_resized = cv2.resize(img, (64, 64))

                if flatten:
                    img_data = img_resized.flatten()
                else:
                    img_data = img_resized.reshape(64, 64, 1)  # For CNNs

                images.append(img_data)
                labels.append(label)

    return np.array(images), np.array(labels)


# ---- Automatic dataset loader ----
def load_dataset(base_folder, flatten=True):
    """
    Loads train, test, val datasets automatically.
    - base_folder: main chest_xray folder (can be "chest_xray" or "chest_xray/chest_xray")
    - flatten: if True, returns flattened images; if False, keeps shape for CNNs
    """
    # Fix for double folder
    double_folder = os.path.join(base_folder, "chest_xray")
    if os.path.isdir(double_folder):
        base_folder = double_folder
    X_train, y_train = load_images_from_folder(os.path.join(base_folder, "train"), flatten)
    X_test, y_test = load_images_from_folder(os.path.join(base_folder, "test"), flatten)
    X_val, y_val = load_images_from_folder(os.path.join(base_folder, "val"), flatten)

    return X_train, y_train, X_test, y_test, X_val, y_val


# ---- Example usage ----
if __name__ == "__main__":
    base_path = "chest_xray"  # Works with or without the extra /chest_xray

    # Flattened for ML models like SVM, logistic regression
    X_train, y_train, X_test, y_test, X_val, y_val = load_dataset(base_path, flatten=True)
    print("Flattened Train set:", X_train.shape, y_train.shape)

    # Non-flattened for CNN models
    X_train_cnn, y_train_cnn, X_test_cnn, y_test_cnn, X_val_cnn, y_val_cnn = load_dataset(base_path, flatten=False)
    print("CNN Train set:", X_train_cnn.shape, y_train_cnn.shape)

Flattened Train set: (5216, 4096) (5216,)
CNN Train set: (5216, 64, 64, 1) (5216,)


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

# Build a simple CNN
cnn_model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(64, 64, 1)),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid')  # 1 output (Pneumonia vs Normal)
])

# Compile the model
cnn_model.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

# Train the model (small number of epochs to start)
history = cnn_model.fit(
    X_train_cnn, y_train_cnn,
    epochs=5,
    batch_size=32,
    validation_data=(X_val_cnn, y_val_cnn)
)


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/5
163/163 ━━━━━━━━━━━━━━━━━━━━ 30s 176ms/step - accuracy: 0.6946 - loss: 9.4184 - val_accuracy: 0.5000 - val_loss: 0.7263
Epoch 2/5
163/163 ━━━━━━━━━━━━━━━━━━━━ 23s 141ms/step - accuracy: 0.7534 - loss: 0.2993 - val_accuracy: 0.8750 - val_loss: 0.4668
Epoch 3/5
163/163 ━━━━━━━━━━━━━━━━━━━━ 41s 142ms/step - accuracy: 0.9470 - loss: 0.2488 - val_accuracy: 0.6875 - val_loss: 0.6418
Epoch 4/5
163/163 ━━━━━━━━━━━━━━━━━━━━ 23s 141ms/step - accuracy: 0.9628 - loss: 0.2214 - val_accuracy: 0.7500 - val_loss: 0.7230
Epoch 5/5
163/163 ━━━━━━━━━━━━━━━━━━━━ 22s 135ms/step - accuracy: 0.9613 - loss: 0.2069 - val_accuracy: 0.8750 - val_loss: 0.4109


In [ ]:
test_loss, test_acc = cnn_model.evaluate(X_test_cnn, y_test_cnn, batch_size=32, verbose=0)
print("Test accuracy:", test_acc)
print("Test loss:", test_loss)

Test accuracy: 0.7483974099159241
Test loss: 1.0041733980178833


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

cnn_model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(64, 64, 1)),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dropout(0.5),              # 👈 new: helps prevent overfitting
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

cnn_model.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

history = cnn_model.fit(
    X_train_cnn, y_train_cnn,
    epochs=15,
    batch_size=32,
    validation_data=(X_val_cnn, y_val_cnn)
)

Epoch 1/15
163/163 ━━━━━━━━━━━━━━━━━━━━ 26s 146ms/step - accuracy: 0.7635 - loss: 7.8162 - val_accuracy: 0.8750 - val_loss: 0.2947
Epoch 2/15
163/163 ━━━━━━━━━━━━━━━━━━━━ 23s 142ms/step - accuracy: 0.9423 - loss: 0.1473 - val_accuracy: 0.8750 - val_loss: 0.3399
Epoch 3/15
163/163 ━━━━━━━━━━━━━━━━━━━━ 41s 145ms/step - accuracy: 0.9486 - loss: 0.1413 - val_accuracy: 0.8750 - val_loss: 0.3682
Epoch 4/15
163/163 ━━━━━━━━━━━━━━━━━━━━ 41s 145ms/step - accuracy: 0.9446 - loss: 0.1510 - val_accuracy: 0.9375 - val_loss: 0.2227
Epoch 5/15
163/163 ━━━━━━━━━━━━━━━━━━━━ 24s 145ms/step - accuracy: 0.9657 - loss: 0.0936 - val_accuracy: 0.8750 - val_loss: 0.3019
Epoch 6/15
163/163 ━━━━━━━━━━━━━━━━━━━━ 41s 144ms/step - accuracy: 0.9681 - loss: 0.0894 - val_accuracy: 0.8750 - val_loss: 0.4103
Epoch 7/15
163/163 ━━━━━━━━━━━━━━━━━━━━ 41s 146ms/step - accuracy: 0.9651 - loss: 0.0976 - val_accuracy: 0.7500 - val_loss: 0.5749
Epoch 8/15
163/163 ━━━━━━━━━━━━━━━━━━━━ 41s 145ms/step - accuracy: 0.9639 - loss: 0

In [ ]:

test_loss, test_acc = cnn_model.evaluate(X_test_cnn, y_test_cnn, batch_size=32, verbose=0)
print("Test accuracy:", test_acc)
print("Test loss:", test_loss)




Test accuracy: 0.7756410241127014
Test loss: 0.9577387571334839


In [ ]:
# ==== CNN with Data Augmentation (copy this whole cell) ====
import tensorflow as tf
from tensorflow.keras import layers, models

# Build a CNN that includes rescaling + augmentation
aug_model = models.Sequential([
    layers.Input(shape=(64, 64, 1)),

    # 1) Make pixel values small (0–1 instead of 0–255)
    layers.Rescaling(1.0/255),

    # 2) Data Augmentation (tiny, safe changes)
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),

    # 3) Convolution blocks
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    # 4) Classifier head
    layers.Flatten(),
    layers.Dropout(0.5),          # helps prevent memorizing
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid')  # binary output (Normal vs Pneumonia)
])

aug_model.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

# Train (give it more time than 5 epochs)
history_aug = aug_model.fit(
    X_train_cnn, y_train_cnn,
    epochs=15,
    batch_size=32,
    validation_data=(X_val_cnn, y_val_cnn),
    verbose=1
)

# Evaluate on the test set
test_loss, test_acc = aug_model.evaluate(X_test_cnn, y_test_cnn, batch_size=32, verbose=0)
print("Augmented Model - Test accuracy:", test_acc)
print("Augmented Model - Test loss:", test_loss)

Epoch 1/15
163/163 ━━━━━━━━━━━━━━━━━━━━ 28s 162ms/step - accuracy: 0.7585 - loss: 0.5130 - val_accuracy: 0.7500 - val_loss: 0.7005
Epoch 2/15
163/163 ━━━━━━━━━━━━━━━━━━━━ 26s 158ms/step - accuracy: 0.9056 - loss: 0.2400 - val_accuracy: 0.8125 - val_loss: 0.4364
Epoch 3/15
163/163 ━━━━━━━━━━━━━━━━━━━━ 27s 168ms/step - accuracy: 0.9080 - loss: 0.2262 - val_accuracy: 0.6250 - val_loss: 1.0909
Epoch 4/15
163/163 ━━━━━━━━━━━━━━━━━━━━ 40s 161ms/step - accuracy: 0.9157 - loss: 0.2092 - val_accuracy: 0.6250 - val_loss: 0.4656
Epoch 5/15
163/163 ━━━━━━━━━━━━━━━━━━━━ 26s 159ms/step - accuracy: 0.9256 - loss: 0.1841 - val_accuracy: 0.6875 - val_loss: 0.5505
Epoch 6/15
163/163 ━━━━━━━━━━━━━━━━━━━━ 25s 156ms/step - accuracy: 0.9351 - loss: 0.1632 - val_accuracy: 0.6875 - val_loss: 0.5603
Epoch 7/15
163/163 ━━━━━━━━━━━━━━━━━━━━ 42s 160ms/step - accuracy: 0.9473 - loss: 0.1478 - val_accuracy: 0.6875 - val_loss: 0.5837
Epoch 8/15
163/163 ━━━━━━━━━━━━━━━━━━━━ 41s 159ms/step - accuracy: 0.9524 - loss: 0

In [ ]:
aug_model.save("cnn_pneumonia_augmented.keras")

In [ ]:
from google.colab import files
files.download("cnn_pneumonia_augmented.keras")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
aug_model.save("/content/drive/MyDrive/cnn_pneumonia_augmented.keras")

In [ ]:
!ls /content/drive/MyDrive | grep cnn_pneumonia

cnn_pneumonia_augmented.keras
